# Notebook-first application walkthrough

**Problem / objective:** Use ordinary linear regression as a transparent baseline for estimating building heating load from physical design variables.

**Decision / solution:** Screen candidate building designs, quantify residual risk, compare regularised/non-linear alternatives and flag high-load configurations for redesign.

This front section is intentionally analysis-first. It uses direct notebook code for inspection, EDA, visualisation and evidence review. The original notebook work is preserved below, followed by modular production code where that adds engineering evidence.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
PROJECT_SLUG = 'linear_regression_energy_efficiency'
ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    candidate = ROOT.parent.parent if ROOT.name == PROJECT_SLUG else ROOT
    if (candidate / 'projects').exists():
        ROOT = candidate
PROJECT = ROOT / 'projects' / PROJECT_SLUG
if not PROJECT.exists() and Path.cwd().name == PROJECT_SLUG:
    PROJECT = Path.cwd()
    ROOT = PROJECT.parent.parent
assert PROJECT.exists(), f'Project directory not found: {PROJECT}'
print('Repository root:', ROOT.resolve())
print('Project:', PROJECT.resolve())


## 1. Find the real data and retained evidence

Instead of hiding the dataset behind a helper function, start by seeing what the project actually ships: raw/small data, fixtures, outputs, results and verified evidence. External large datasets remain reproducibly downloadable from the documented source.


In [ ]:
candidate_files = []
for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
    candidate_files.extend(PROJECT.rglob(pattern))
verified_dir = ROOT / 'verified' / PROJECT_SLUG
if verified_dir.exists():
    for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
        candidate_files.extend(verified_dir.rglob(pattern))
candidate_files = sorted({p.resolve() for p in candidate_files if p.is_file()})
file_inventory = pd.DataFrame({
    'file': [str(p.relative_to(ROOT)) if ROOT in p.parents else str(p) for p in candidate_files],
    'suffix': [p.suffix.lower() for p in candidate_files],
    'size_kb': [round(p.stat().st_size / 1024, 1) for p in candidate_files],
})
display(file_inventory.head(40))
print(f'Inspectable local data/evidence files: {len(file_inventory):,}')


## 2. Direct tabular data audit

The code below deliberately avoids a project-specific wrapper. It opens the first sensible local tabular asset, shows its schema and quality profile, and makes the data issues visible before modelling. If the full raw dataset is external, run the project's documented download cell/entry point first and rerun this section.


In [ ]:
tabular_candidates = [p for p in candidate_files if p.suffix.lower() in {'.csv', '.tsv', '.parquet'}]
preferred = [p for p in tabular_candidates if not any(token in p.name.lower() for token in ('metric', 'summary', 'verification'))]
tabular_path = (preferred or tabular_candidates or [None])[0]
df = None
if tabular_path is not None:
    if tabular_path.suffix.lower() == '.parquet':
        df = pd.read_parquet(tabular_path)
    else:
        sep = '\t' if tabular_path.suffix.lower() == '.tsv' else ','
        df = pd.read_csv(tabular_path, sep=sep, nrows=200_000)
    print('Loaded:', tabular_path)
    print('Shape:', df.shape)
    display(df.head())
    audit = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing': df.isna().sum(),
        'missing_pct': (100 * df.isna().mean()).round(2),
        'unique': df.nunique(dropna=False),
    }).sort_values(['missing_pct', 'unique'], ascending=[False, False])
    display(audit.head(30))
    print('Duplicate rows:', int(df.duplicated().sum()))
else:
    print('No local CSV/TSV/Parquet found yet. Use the project README/run path to download or build the documented dataset, then rerun this audit.')


## 3. Exploratory data analysis and visualisation

These plots are intentionally created in the notebook rather than described in prose. They expose distribution, missingness, scale, category balance and numeric relationships before any final model decision.


In [ ]:
if df is not None and len(df):
    missing_pct = (100 * df.isna().mean()).sort_values(ascending=False).head(20)
    missing_pct = missing_pct[missing_pct > 0]
    if len(missing_pct):
        plt.figure(figsize=(10, 4))
        missing_pct.plot(kind='bar')
        plt.title('Missing values by feature (%)')
        plt.ylabel('Missing %')
        plt.xticks(rotation=60, ha='right')
        plt.tight_layout()
        plt.show()

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:8]
    for col in numeric_cols:
        series = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(series):
            plt.figure(figsize=(8, 4))
            plt.hist(series, bins=30, alpha=0.8)
            plt.axvline(series.median(), linestyle='--', label=f'median={series.median():.2f}')
            plt.title(f'Distribution: {col}')
            plt.xlabel(col)
            plt.ylabel('Count')
            plt.legend()
            plt.tight_layout()
            plt.show()

    categorical_cols = [c for c in df.columns if c not in numeric_cols and df[c].nunique(dropna=False) <= 30][:4]
    for col in categorical_cols:
        counts = df[col].fillna('<missing>').astype(str).value_counts().head(15)
        plt.figure(figsize=(9, 4))
        counts.sort_values().plot(kind='barh')
        plt.title(f'Top categories: {col}')
        plt.xlabel('Rows')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr(numeric_only=True)
        plt.figure(figsize=(8, 6))
        image = plt.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
        plt.colorbar(image, label='Correlation')
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=60, ha='right')
        plt.yticks(range(len(corr.index)), corr.index)
        plt.title('Numeric correlation matrix')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        x_col, y_col = numeric_cols[0], numeric_cols[-1]
        sample = df[[x_col, y_col]].dropna().sample(min(3000, len(df.dropna(subset=[x_col, y_col]))), random_state=42)
        if len(sample):
            plt.figure(figsize=(7, 5))
            plt.scatter(sample[x_col], sample[y_col], alpha=0.35, s=18)
            plt.xlabel(x_col)
            plt.ylabel(y_col)
            plt.title(f'{y_col} versus {x_col}')
            plt.tight_layout()
            plt.show()
else:
    print('Run the documented data-build/download path, then rerun this section to render raw-data EDA.')


## 4. Inspect the measured results, not just the code

A portfolio project is stronger when it retains evidence. This section reads machine-readable JSON/CSV outputs and turns scalar metrics into a quick visual comparison.


In [ ]:
json_files = [p for p in candidate_files if p.suffix.lower() == '.json']
metric_rows = []
for path in json_files[:30]:
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        continue
    stack = [('', payload)]
    while stack:
        prefix, value = stack.pop()
        if isinstance(value, dict):
            for key, child in value.items():
                stack.append((f'{prefix}.{key}' if prefix else str(key), child))
        elif isinstance(value, (int, float)) and not isinstance(value, bool) and np.isfinite(value):
            metric_rows.append({
                'file': str(path.relative_to(ROOT)) if ROOT in path.parents else str(path),
                'metric': prefix,
                'value': float(value),
            })
metrics_df = pd.DataFrame(metric_rows)
if len(metrics_df):
    display(metrics_df.head(40))
    plot_df = metrics_df[np.isfinite(metrics_df['value'])].copy()
    plot_df = plot_df[plot_df['value'].abs() < 1_000_000].head(20)
    if len(plot_df):
        labels = (plot_df['file'].str.split('/').str[-1] + ' :: ' + plot_df['metric']).tolist()
        plt.figure(figsize=(10, max(4, 0.35 * len(plot_df))))
        plt.barh(range(len(plot_df)), plot_df['value'])
        plt.yticks(range(len(plot_df)), labels)
        plt.title('Retained project metrics / evidence')
        plt.tight_layout()
        plt.show()
else:
    print('No scalar JSON evidence found. Run the project and retain metrics/results before treating it as complete.')


## 5. Reproduce the application

The notebook should be understandable without running anything, but a reviewer can reproduce the canonical application below. The switch is off by default so opening the notebook never triggers a long training job unexpectedly.


In [ ]:
RUN_PROJECT = False
entrypoint = PROJECT / 'run.py'
if RUN_PROJECT and entrypoint.exists():
    subprocess.run([sys.executable, str(entrypoint)], cwd=PROJECT, check=True)
elif entrypoint.exists():
    print(f'Reproduce with: cd {PROJECT} && {sys.executable} run.py')
else:
    print('This project uses a different documented entry point; see README.md in the project folder.')


## 6. Decision / solution

Screen candidate building designs, quantify residual risk, compare regularised/non-linear alternatives and flag high-load configurations for redesign.

The final recommendation should be tied to the measured validation evidence and error analysis below. A model is not the solution by itself; the solution is the decision process built around it.


# Linear Regression — Building Energy Efficiency Decision Model

**Goal:** estimate building heating load transparently, compare ordinary Linear Regression with sensible alternatives, visualise the data and residuals, and turn predictions into a simple design-review decision.

**Dataset:** UCI Energy Efficiency (768 building configurations, 8 design features, CC BY 4.0, DOI 10.24432/C51307).


## 1. Load the real dataset and inspect it directly
The notebook deliberately shows the analysis instead of hiding everything in helper functions.


In [ ]:
from pathlib import Path
import json, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
warnings.filterwarnings('ignore')
SEED = 42
rng = np.random.default_rng(SEED)
energy = fetch_ucirepo(id=242)
X_raw = energy.data.features.copy()
y_raw = energy.data.targets.copy()
feature_names = ['relative_compactness','surface_area','wall_area','roof_area','overall_height','orientation','glazing_area','glazing_area_distribution']
target_names = ['heating_load','cooling_load']
X_raw.columns = feature_names
y_raw = y_raw.iloc[:, :2].copy()
y_raw.columns = target_names
df = pd.concat([X_raw, y_raw], axis=1)
print('Shape:', df.shape)
display(df.head())
display(df.describe().T.round(3))
audit = pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(), 'unique': df.nunique(dropna=False)})
display(audit)
print('Duplicate rows:', int(df.duplicated().sum()))


## 2. EDA and visualisation
Before modelling, inspect distributions, relationships, categories and correlation.


In [ ]:
plt.figure(figsize=(8,4))
plt.hist(df['heating_load'], bins=28, alpha=0.8)
plt.axvline(df['heating_load'].median(), linestyle='--', label=f"median={df['heating_load'].median():.2f}")
plt.title('Heating-load distribution')
plt.xlabel('Heating Load')
plt.ylabel('Buildings')
plt.legend()
plt.tight_layout()
plt.show()
plt.figure(figsize=(7,5))
plt.scatter(df['heating_load'], df['cooling_load'], alpha=0.45, s=22)
plt.xlabel('Heating Load')
plt.ylabel('Cooling Load')
plt.title('Heating vs cooling load')
plt.tight_layout()
plt.show()
for col in ['relative_compactness','surface_area','wall_area','roof_area','overall_height','glazing_area']:
    plt.figure(figsize=(7,4))
    plt.scatter(df[col], df['heating_load'], alpha=0.40, s=20)
    plt.xlabel(col)
    plt.ylabel('Heating Load')
    plt.title(f'Heating Load vs {col}')
    plt.tight_layout()
    plt.show()
orientation_view = df.groupby('orientation', as_index=False)['heating_load'].agg(['mean','median','count']).reset_index()
display(orientation_view.round(3))
plt.figure(figsize=(7,4))
plt.bar(orientation_view['orientation'].astype(str), orientation_view['mean'])
plt.xlabel('Orientation')
plt.ylabel('Mean Heating Load')
plt.title('Heating load by orientation')
plt.tight_layout()
plt.show()
glazing_view = df.groupby('glazing_area', as_index=False)['heating_load'].agg(['mean','median','count']).reset_index()
display(glazing_view.round(3))
plt.figure(figsize=(7,4))
plt.bar(glazing_view['glazing_area'].astype(str), glazing_view['mean'])
plt.xlabel('Glazing area')
plt.ylabel('Mean Heating Load')
plt.title('Heating load by glazing area')
plt.tight_layout()
plt.show()
corr = df.corr(numeric_only=True)
plt.figure(figsize=(9,7))
image = plt.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
plt.colorbar(image, label='Correlation')
plt.xticks(range(len(corr.columns)), corr.columns, rotation=70, ha='right')
plt.yticks(range(len(corr.index)), corr.index)
plt.title('Correlation matrix')
plt.tight_layout()
plt.show()


## 3. Baseline, ordinary Linear Regression and alternatives
Orientation and glazing-distribution are treated as categories. Numeric variables are imputed/scaled inside the pipeline to keep preprocessing leakage-safe.


In [ ]:
features = feature_names
target = 'heating_load'
categorical = ['orientation','glazing_area_distribution']
numeric = [c for c in features if c not in categorical]
X = df[features].copy()
y = df[target].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED)
numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
categorical_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocessor = ColumnTransformer([('numeric', numeric_pipe, numeric), ('categorical', categorical_pipe, categorical)])
baseline = DummyRegressor(strategy='median').fit(X_train, y_train)
linear = Pipeline([('preprocess', preprocessor), ('model', LinearRegression())]).fit(X_train, y_train)
ridge = Pipeline([('preprocess', preprocessor), ('model', RidgeCV(alphas=np.logspace(-4,4,80)))]).fit(X_train, y_train)
lasso = Pipeline([('preprocess', preprocessor), ('model', LassoCV(alphas=np.logspace(-4,1,80), cv=5, random_state=SEED, max_iter=100000))]).fit(X_train, y_train)
poly_pre = ColumnTransformer([('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('poly', PolynomialFeatures(degree=2, include_bias=False)), ('scale', StandardScaler())]), numeric), ('categorical', categorical_pipe, categorical)])
poly = Pipeline([('preprocess', poly_pre), ('model', RidgeCV(alphas=np.logspace(-4,4,80)))]).fit(X_train, y_train)
models = {'dummy_median': baseline, 'linear_regression': linear, 'ridge': ridge, 'lasso': lasso, 'polynomial_ridge': poly}
rows = []
predictions = {}
for name, model in models.items():
    pred = model.predict(X_test)
    predictions[name] = pred
    rows.append({'model': name, 'mae': mean_absolute_error(y_test,pred), 'rmse': math.sqrt(mean_squared_error(y_test,pred)), 'r2': r2_score(y_test,pred)})
metrics = pd.DataFrame(rows).sort_values('rmse')
display(metrics.round(4))
plt.figure(figsize=(8,4))
plt.bar(metrics['model'], metrics['rmse'])
plt.ylabel('RMSE')
plt.title('Holdout model comparison')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()
best_name = metrics.iloc[0]['model']
print('Best holdout model:', best_name)
print('Ridge alpha:', ridge.named_steps['model'].alpha_)
print('Lasso alpha:', lasso.named_steps['model'].alpha_)


## 4. Cross-validation and residual/error analysis
A strong project does not stop at one holdout score. Compare folds and inspect where the transparent linear model is wrong.


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
cv_rows = []
for name, model in {'linear_regression': linear, 'ridge': ridge, 'polynomial_ridge': poly}.items():
    fold_scores = -cross_val_score(model, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error')
    cv_rows.append({'model': name, 'cv_rmse_mean': fold_scores.mean(), 'cv_rmse_std': fold_scores.std()})
cv_results = pd.DataFrame(cv_rows).sort_values('cv_rmse_mean')
display(cv_results.round(4))
linear_pred = predictions['linear_regression']
result = X_test.reset_index(drop=False).rename(columns={'index':'source_index'})
result['actual'] = y_test.reset_index(drop=True)
result['predicted'] = linear_pred
result['residual'] = result['actual'] - result['predicted']
result['abs_error'] = result['residual'].abs()
display(result.nlargest(12, 'abs_error').round(3))
plt.figure(figsize=(7,5))
plt.scatter(result['actual'], result['predicted'], alpha=0.60, s=25)
lo = min(result['actual'].min(), result['predicted'].min())
hi = max(result['actual'].max(), result['predicted'].max())
plt.plot([lo,hi],[lo,hi], linestyle='--')
plt.xlabel('Actual Heating Load')
plt.ylabel('Linear Regression prediction')
plt.title('Actual vs predicted')
plt.tight_layout()
plt.show()
plt.figure(figsize=(7,5))
plt.scatter(result['predicted'], result['residual'], alpha=0.60, s=25)
plt.axhline(0, linestyle='--')
plt.xlabel('Predicted')
plt.ylabel('Residual')
plt.title('Residuals vs predicted')
plt.tight_layout()
plt.show()
plt.figure(figsize=(7,4))
plt.hist(result['residual'], bins=25, alpha=0.80)
plt.axvline(0, linestyle='--')
plt.xlabel('Residual')
plt.ylabel('Rows')
plt.title('Residual distribution')
plt.tight_layout()
plt.show()
orientation_error = result.groupby('orientation', as_index=False)['abs_error'].agg(['mean','median','count']).reset_index()
display(orientation_error.round(4))
plt.figure(figsize=(7,4))
plt.bar(orientation_error['orientation'].astype(str), orientation_error['mean'])
plt.xlabel('Orientation')
plt.ylabel('MAE')
plt.title('Linear Regression error by orientation')
plt.tight_layout()
plt.show()


## 5. Coefficients, uncertainty and the decision layer
Coefficients are useful for transparency but are not causal effects. Correlated physical design variables make that distinction important.


In [ ]:
prep = linear.named_steps['preprocess']
ols = linear.named_steps['model']
cat_names = prep.named_transformers_['categorical'].named_steps['onehot'].get_feature_names_out(categorical).tolist()
names = numeric + cat_names
coef = pd.DataFrame({'feature': names, 'coefficient': ols.coef_})
coef['abs_coefficient'] = coef['coefficient'].abs()
coef = coef.sort_values('abs_coefficient', ascending=False)
display(coef.round(4))
plot_coef = coef.head(15).sort_values('coefficient')
plt.figure(figsize=(8,5))
plt.barh(plot_coef['feature'], plot_coef['coefficient'])
plt.xlabel('Coefficient')
plt.title('Linear Regression coefficients')
plt.tight_layout()
plt.show()
bootstrap_predictions = []
bootstrap_rmse = []
for i in range(150):
    positions = rng.integers(0, len(X_train), size=len(X_train))
    X_boot = X_train.iloc[positions]
    y_boot = y_train.iloc[positions]
    boot = Pipeline([('preprocess', preprocessor), ('model', LinearRegression())]).fit(X_boot, y_boot)
    pred = boot.predict(X_test)
    bootstrap_predictions.append(pred)
    bootstrap_rmse.append(math.sqrt(mean_squared_error(y_test, pred)))
bootstrap_predictions = np.vstack(bootstrap_predictions)
result['p05'] = np.quantile(bootstrap_predictions, 0.05, axis=0)
result['p50'] = np.quantile(bootstrap_predictions, 0.50, axis=0)
result['p95'] = np.quantile(bootstrap_predictions, 0.95, axis=0)
result['interval_width'] = result['p95'] - result['p05']
print('Bootstrap RMSE mean:', np.mean(bootstrap_rmse))
print('Bootstrap RMSE 5%-95%:', np.quantile(bootstrap_rmse, [0.05,0.95]))
print('Mean prediction interval width:', result['interval_width'].mean())
plt.figure(figsize=(7,4))
plt.hist(bootstrap_rmse, bins=25, alpha=0.8)
plt.xlabel('Bootstrap RMSE')
plt.ylabel('Refits')
plt.title('Bootstrap uncertainty')
plt.tight_layout()
plt.show()
review_threshold = y_train.quantile(0.75)
result['decision'] = np.where(result['predicted'] >= review_threshold, 'HIGH LOAD - REVIEW', 'LOW / NORMAL LOAD')
display(result[['actual','predicted','p05','p95','decision']].head(20).round(3))
print('Review threshold:', review_threshold)
print(result['decision'].value_counts())


## Conclusion / solution
Ordinary Linear Regression is retained as the transparent baseline and screening model. The project compares it with regularised and polynomial alternatives rather than assuming complexity is automatically better. Predictions above the training 75th-percentile heating-load threshold are flagged for design review, with bootstrap uncertainty shown alongside the estimate.

## Limitations and next steps
The UCI dataset is simulated and small. Coefficients are associations, not causal effects. A production building-energy workflow would require real measured buildings, climate/location variables, external validation, stronger uncertainty modelling and domain-engineering review.


## Reproducibility
Run `python run.py` from this project folder to recreate metrics, tables, models and PNG visualisations in `results/`. The engineering appendix added by the portfolio sync exposes the complete canonical application source inside this notebook as well.


# Deeper exploratory analysis and retained evidence

These direct notebook cells extend the initial EDA with data-quality, scale, relationship, output and error diagnostics. They are intentionally visible here rather than hidden behind project helper functions.


In [ ]:
# Extended data-quality scorecard
if df is not None and len(df):
    quality_rows = []
    for col in df.columns:
        series = df[col]
        row = {
            'feature': col,
            'dtype': str(series.dtype),
            'rows': len(series),
            'missing': int(series.isna().sum()),
            'missing_pct': float(100 * series.isna().mean()),
            'unique': int(series.nunique(dropna=False)),
            'unique_pct': float(100 * series.nunique(dropna=False) / max(len(series), 1)),
        }
        if pd.api.types.is_numeric_dtype(series):
            values = pd.to_numeric(series, errors='coerce').dropna()
            if len(values):
                q1, q3 = values.quantile([0.25, 0.75])
                iqr = q3 - q1
                row.update({
                    'mean': float(values.mean()),
                    'median': float(values.median()),
                    'std': float(values.std()),
                    'p05': float(values.quantile(0.05)),
                    'p95': float(values.quantile(0.95)),
                    'skew': float(values.skew()),
                    'iqr_outliers': int(((values < q1 - 1.5*iqr) | (values > q3 + 1.5*iqr)).sum()),
                })
        quality_rows.append(row)
    deep_quality = pd.DataFrame(quality_rows)
    display(deep_quality.sort_values(['missing_pct','unique'], ascending=[False,False]).head(40))
    if 'iqr_outliers' in deep_quality:
        outlier_view = deep_quality.dropna(subset=['iqr_outliers']).sort_values('iqr_outliers', ascending=False).head(15)
        if len(outlier_view):
            plt.figure(figsize=(10,4))
            plt.bar(outlier_view['feature'], outlier_view['iqr_outliers'])
            plt.title('Potential IQR outliers by feature')
            plt.ylabel('Rows')
            plt.xticks(rotation=60, ha='right')
            plt.tight_layout()
            plt.show()
    card = deep_quality.sort_values('unique', ascending=False).head(20)
    plt.figure(figsize=(10,4))
    plt.bar(card['feature'], card['unique'])
    plt.title('Feature cardinality')
    plt.ylabel('Unique values')
    plt.xticks(rotation=60, ha='right')
    plt.tight_layout()
    plt.show()
    print('Constant columns:', deep_quality.loc[deep_quality['unique'] <= 1, 'feature'].tolist())
    print('High-missing columns:', deep_quality.loc[deep_quality['missing_pct'] >= 30, 'feature'].tolist())
    print('Possible identifier columns:', deep_quality.loc[deep_quality['unique_pct'] >= 95, 'feature'].tolist()[:20])
else:
    print('Materialise the documented dataset to run the extended data-quality scorecard.')


In [ ]:
# Numeric distributions, spread and strongest pairwise relationships
if df is not None and len(df):
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:12]
    for col in numeric_cols:
        values = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(values) < 5:
            continue
        clipped = values.clip(values.quantile(0.01), values.quantile(0.99))
        plt.figure(figsize=(8,4))
        plt.hist(clipped, bins=35, alpha=0.82)
        plt.axvline(values.median(), linestyle='--', label=f'median={values.median():.3g}')
        plt.axvline(values.mean(), linestyle=':', label=f'mean={values.mean():.3g}')
        plt.title(f'Distribution: {col} (1st–99th percentile)')
        plt.xlabel(col)
        plt.ylabel('Rows')
        plt.legend()
        plt.tight_layout()
        plt.show()
        plt.figure(figsize=(8,3))
        plt.boxplot(values, vert=False, showfliers=True)
        plt.title(f'Spread / outliers: {col}')
        plt.xlabel(col)
        plt.tight_layout()
        plt.show()
    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr(numeric_only=True)
        pairs = []
        for i, left in enumerate(corr.columns):
            for right in corr.columns[i+1:]:
                value = corr.loc[left, right]
                if pd.notna(value):
                    pairs.append({'feature_a': left, 'feature_b': right, 'correlation': float(value), 'abs_correlation': float(abs(value))})
        corr_pairs = pd.DataFrame(pairs).sort_values('abs_correlation', ascending=False) if pairs else pd.DataFrame()
        if len(corr_pairs):
            display(corr_pairs.head(20).round(4))
            for _, pair in corr_pairs.head(4).iterrows():
                sample = df[[pair['feature_a'], pair['feature_b']]].dropna()
                if len(sample) > 3000:
                    sample = sample.sample(3000, random_state=42)
                plt.figure(figsize=(7,5))
                plt.scatter(sample[pair['feature_a']], sample[pair['feature_b']], alpha=0.30, s=16)
                plt.xlabel(pair['feature_a'])
                plt.ylabel(pair['feature_b'])
                plt.title(f"{pair['feature_a']} vs {pair['feature_b']} (r={pair['correlation']:.2f})")
                plt.tight_layout()
                plt.show()
    categorical = [c for c in df.columns if 2 <= df[c].nunique(dropna=False) <= 20][:8]
    for col in categorical:
        counts = df[col].fillna('<missing>').astype(str).value_counts().head(20)
        shares = 100 * counts / counts.sum()
        display(pd.DataFrame({'rows': counts, 'share_pct': shares.round(2)}))
        plt.figure(figsize=(8,4))
        counts.sort_values().plot(kind='barh')
        plt.title(f'Category balance: {col}')
        plt.xlabel('Rows')
        plt.tight_layout()
        plt.show()
else:
    print('Materialise the documented dataset to run distribution diagnostics.')


In [ ]:
# Temporal coverage where date/time fields exist
if df is not None and len(df):
    time_cols = [c for c in df.columns if any(token in str(c).lower() for token in ('date','time','timestamp','datetime'))]
    print('Date/time candidates:', time_cols[:10])
    for col in time_cols[:4]:
        converted = pd.to_datetime(df[col], errors='coerce')
        valid = converted.dropna()
        if len(valid) >= max(10, int(0.25*len(df))):
            print(col, 'range:', valid.min(), '→', valid.max())
            monthly = valid.dt.to_period('M').value_counts().sort_index()
            if len(monthly) > 1:
                plt.figure(figsize=(10,4))
                plt.plot(monthly.index.astype(str), monthly.values, marker='o')
                plt.title(f'Rows over time: {col}')
                plt.ylabel('Rows')
                plt.xticks(rotation=70, ha='right')
                plt.tight_layout()
                plt.show()


## Retained outputs and error analysis

A strong portfolio keeps inspectable evidence. The cells below profile compact result tables and automatically detect prediction-like columns for residual or misclassification analysis.


In [ ]:
# Load compact result/evidence tables
result_tables = []
for base in [PROJECT/'results', PROJECT/'outputs', PROJECT/'artifacts', ROOT/'verified'/PROJECT_SLUG]:
    if not base.exists():
        continue
    for path in sorted(base.rglob('*')):
        if path.is_file() and path.suffix.lower() in {'.csv','.tsv','.parquet'} and path.stat().st_size < 25_000_000:
            try:
                if path.suffix.lower() == '.parquet':
                    table = pd.read_parquet(path)
                else:
                    table = pd.read_csv(path, sep='	' if path.suffix.lower() == '.tsv' else ',')
            except Exception as exc:
                print('Could not read', path.name, '-', exc)
                continue
            result_tables.append((path, table))
            print('
RESULT TABLE:', path.relative_to(ROOT) if ROOT in path.parents else path)
            print('shape=', table.shape)
            display(table.head(15))
            numeric = table.select_dtypes(include=np.number).columns.tolist()[:12]
            if numeric:
                display(table[numeric].describe().T.round(4))
print('Inspectable result tables:', len(result_tables))


In [ ]:
# Automatic regression/classification-style error diagnostics
actual_tokens = ('actual','target','truth','y_true','observed','label')
pred_tokens = ('prediction','predicted','forecast','y_pred')
confidence_tokens = ('confidence','probability','proba','risk','uncertainty')
for path, table in result_tables:
    actual_cols = [c for c in table.columns if any(token in str(c).lower() for token in actual_tokens)]
    pred_cols = [c for c in table.columns if any(token in str(c).lower() for token in pred_tokens)]
    conf_cols = [c for c in table.columns if any(token in str(c).lower() for token in confidence_tokens)]
    if actual_cols and pred_cols and len(table):
        actual_col = actual_cols[0]
        pred_col = next((c for c in pred_cols if c != actual_col), pred_cols[0])
        actual_num = pd.to_numeric(table[actual_col], errors='coerce')
        pred_num = pd.to_numeric(table[pred_col], errors='coerce')
        numeric_mask = actual_num.notna() & pred_num.notna()
        if numeric_mask.sum() >= 10:
            residual = actual_num[numeric_mask] - pred_num[numeric_mask]
            abs_error = residual.abs()
            print('
', path.name, '| MAE=', round(float(abs_error.mean()),5), '| RMSE=', round(float(np.sqrt(np.mean(residual**2))),5), '| bias=', round(float(residual.mean()),5))
            plt.figure(figsize=(7,5))
            plt.scatter(actual_num[numeric_mask], pred_num[numeric_mask], alpha=0.35, s=18)
            lo = min(actual_num[numeric_mask].min(), pred_num[numeric_mask].min())
            hi = max(actual_num[numeric_mask].max(), pred_num[numeric_mask].max())
            plt.plot([lo,hi],[lo,hi], linestyle='--')
            plt.xlabel(str(actual_col))
            plt.ylabel(str(pred_col))
            plt.title(f'Actual vs predicted — {path.name}')
            plt.tight_layout()
            plt.show()
            plt.figure(figsize=(7,4))
            plt.hist(residual, bins=30, alpha=0.82)
            plt.axvline(0, linestyle='--')
            plt.title(f'Residual distribution — {path.name}')
            plt.tight_layout()
            plt.show()
            worst_idx = abs_error.nlargest(min(15,len(abs_error))).index
            cols = list(dict.fromkeys([actual_col,pred_col]+conf_cols[:2]))
            worst = table.loc[worst_idx, cols].copy()
            worst['absolute_error'] = abs_error.loc[worst_idx].values
            display(worst.sort_values('absolute_error', ascending=False))
        else:
            agreement = table[actual_col].astype(str) == table[pred_col].astype(str)
            print('
', path.name, '| classification agreement=', round(float(agreement.mean()),4))
            if (~agreement).any():
                display(table.loc[~agreement, [actual_col,pred_col]+conf_cols[:2]].head(20))
    elif conf_cols:
        for col in conf_cols[:2]:
            values = pd.to_numeric(table[col], errors='coerce').dropna()
            if len(values) >= 10:
                plt.figure(figsize=(7,4))
                plt.hist(values, bins=30, alpha=0.82)
                plt.title(f'{col} distribution — {path.name}')
                plt.tight_layout()
                plt.show()


In [ ]:
# Display retained visual evidence from actual project runs
png_files = []
for base in [PROJECT/'results', PROJECT/'outputs', PROJECT/'artifacts', ROOT/'verified'/PROJECT_SLUG]:
    if base.exists():
        png_files.extend(sorted(base.rglob('*.png')))
print('Retained PNG figures:', len(png_files))
for path in png_files[:12]:
    try:
        image = plt.imread(path)
        plt.figure(figsize=(10,6))
        plt.imshow(image)
        plt.axis('off')
        plt.title(str(path.relative_to(ROOT)) if ROOT in path.parents else path.name)
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print('Could not display', path.name, '-', exc)


In [ ]:
# Reproducibility and evidence checklist
checks = [
    {'check':'README present', 'status':(PROJECT/'README.md').exists()},
    {'check':'Recruiter notebook present', 'status':(PROJECT/'project_notebook.ipynb').exists()},
    {'check':'Python implementation present', 'status':any(PROJECT.rglob('*.py'))},
    {'check':'Tests present', 'status':(PROJECT/'tests').exists() and any((PROJECT/'tests').rglob('test*.py'))},
    {'check':'Result/evidence files present', 'status':bool(candidate_files)},
    {'check':'Machine-readable JSON evidence', 'status':bool(json_files)},
    {'check':'Retained visual evidence', 'status':bool(png_files)},
]
checklist = pd.DataFrame(checks)
display(checklist)
print('Evidence checklist pass rate:', f"{100*checklist['status'].mean():.1f}%")
print('A failed item is a prompt to strengthen the project, not something to hide.')


# Robustness, slices and decision analysis

A model or pipeline is useful only when we know where it works, where it fails and what action follows. This section adds direct slice analysis, sensitivity checks and a compact decision memo from the evidence already produced by the project.


In [ ]:
# Quantile slices for important numeric variables
if df is not None and len(df):
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:10]
    quantile_rows = []
    for col in numeric_cols:
        values = pd.to_numeric(df[col], errors='coerce')
        valid = values.dropna()
        if len(valid) < 20 or valid.nunique() < 5:
            continue
        quantiles = valid.quantile([0.01,0.05,0.10,0.25,0.50,0.75,0.90,0.95,0.99])
        for q, value in quantiles.items():
            quantile_rows.append({'feature':col, 'quantile':q, 'value':float(value)})
    quantile_table = pd.DataFrame(quantile_rows)
    if len(quantile_table):
        display(quantile_table.pivot(index='feature', columns='quantile', values='value').round(4))
        for col in quantile_table['feature'].unique()[:6]:
            view = quantile_table[quantile_table['feature']==col]
            plt.figure(figsize=(7,4))
            plt.plot(view['quantile'], view['value'], marker='o')
            plt.xlabel('Quantile')
            plt.ylabel(col)
            plt.title(f'Quantile profile: {col}')
            plt.tight_layout()
            plt.show()
else:
    print('Quantile slices become available after the project dataset is materialised.')


In [ ]:
# Missingness and duplication sensitivity
if df is not None and len(df):
    missing_by_row = df.isna().sum(axis=1)
    print('Rows with any missing value:', int((missing_by_row>0).sum()))
    print('Rows with 2+ missing values:', int((missing_by_row>=2).sum()))
    print('Exact duplicate rows:', int(df.duplicated().sum()))
    if missing_by_row.max() > 0:
        plt.figure(figsize=(7,4))
        missing_by_row.value_counts().sort_index().plot(kind='bar')
        plt.title('Missing cells per row')
        plt.xlabel('Missing cells')
        plt.ylabel('Rows')
        plt.tight_layout()
        plt.show()
    duplicated = df.duplicated(keep=False)
    if duplicated.any():
        display(df.loc[duplicated].head(20))
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:10]
    robust_rows = []
    for col in numeric_cols:
        values = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(values) < 20:
            continue
        median = values.median()
        mad = np.median(np.abs(values-median))
        robust_z = 0.6745*(values-median)/(mad if mad else 1.0)
        robust_rows.append({'feature':col, 'median':median, 'mad':mad, 'robust_outliers_abs_z_gt_3_5':int((np.abs(robust_z)>3.5).sum())})
    robust_outliers = pd.DataFrame(robust_rows).sort_values('robust_outliers_abs_z_gt_3_5', ascending=False) if robust_rows else pd.DataFrame()
    if len(robust_outliers):
        display(robust_outliers.round(4))


In [ ]:
# Concentration / imbalance analysis for important categorical dimensions
if df is not None and len(df):
    categorical = [c for c in df.columns if 2 <= df[c].nunique(dropna=False) <= 50][:10]
    concentration_rows = []
    for col in categorical:
        counts = df[col].fillna('<missing>').astype(str).value_counts()
        shares = counts / counts.sum()
        hhi = float((shares**2).sum())
        concentration_rows.append({'feature':col, 'categories':len(counts), 'largest_share':float(shares.iloc[0]), 'top3_share':float(shares.head(3).sum()), 'hhi':hhi})
    concentration = pd.DataFrame(concentration_rows).sort_values('hhi', ascending=False) if concentration_rows else pd.DataFrame()
    if len(concentration):
        display(concentration.round(4))
        plt.figure(figsize=(9,4))
        plt.bar(concentration['feature'], concentration['largest_share'])
        plt.ylabel('Largest category share')
        plt.title('Category concentration / imbalance')
        plt.xticks(rotation=60, ha='right')
        plt.tight_layout()
        plt.show()


In [ ]:
# Rank all retained scalar metrics and highlight likely success/risk signals
metric_records = []
for path in json_files[:60]:
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        continue
    stack = [('', payload)]
    while stack:
        prefix, value = stack.pop()
        if isinstance(value, dict):
            for key, child in value.items():
                stack.append((f'{prefix}.{key}' if prefix else str(key), child))
        elif isinstance(value, (int,float)) and not isinstance(value,bool) and np.isfinite(value):
            metric_records.append({'file':path.name, 'metric':prefix, 'value':float(value)})
all_metrics = pd.DataFrame(metric_records)
if len(all_metrics):
    signal_pattern = 'accuracy|f1|auc|precision|recall|r2|rmse|mae|loss|coverage|review|drift|psi|brier|calibration|revenue|cost|effect|lift|latency|row|reject|duplicate'
    decision_metrics = all_metrics[all_metrics['metric'].str.contains(signal_pattern, case=False, regex=True)].copy()
    if not len(decision_metrics):
        decision_metrics = all_metrics.copy()
    decision_metrics = decision_metrics.drop_duplicates(['file','metric']).reset_index(drop=True)
    display(decision_metrics.head(60).round(6))
    rate_like = decision_metrics[decision_metrics['metric'].str.contains('accuracy|f1|auc|precision|recall|coverage|rate|r2', case=False, regex=True)]
    if len(rate_like):
        bounded = rate_like[(rate_like['value']>=-1)&(rate_like['value']<=1)].head(30)
        if len(bounded):
            plt.figure(figsize=(10,max(5,0.3*len(bounded))))
            plt.barh(range(len(bounded)), bounded['value'])
            plt.yticks(range(len(bounded)), bounded['file']+' :: '+bounded['metric'])
            plt.xlim(min(-0.05,bounded['value'].min()-0.05),1.05)
            plt.title('Retained rate / quality metrics')
            plt.tight_layout()
            plt.show()
    error_like = decision_metrics[decision_metrics['metric'].str.contains('rmse|mae|loss|error|latency|drift|psi|brier', case=False, regex=True)]
    if len(error_like):
        display(error_like.sort_values('value', ascending=False).head(30).round(6))
else:
    print('No retained scalar JSON metrics are available yet.')


In [ ]:
# Inspect artifact sizes — a quick engineering sanity check
artifact_rows = []
for base in [PROJECT/'artifacts', PROJECT/'results', PROJECT/'outputs', ROOT/'verified'/PROJECT_SLUG]:
    if not base.exists():
        continue
    for path in base.rglob('*'):
        if path.is_file():
            artifact_rows.append({'file':str(path.relative_to(ROOT)) if ROOT in path.parents else str(path), 'suffix':path.suffix.lower(), 'size_kb':path.stat().st_size/1024})
artifacts_df = pd.DataFrame(artifact_rows).sort_values('size_kb', ascending=False) if artifact_rows else pd.DataFrame()
if len(artifacts_df):
    display(artifacts_df.head(40).round(2))
    by_type = artifacts_df.groupby('suffix', as_index=False).agg(files=('file','size'), total_kb=('size_kb','sum')).sort_values('total_kb', ascending=False)
    display(by_type.round(2))
    plt.figure(figsize=(8,4))
    plt.bar(by_type['suffix'].replace('', '<none>'), by_type['total_kb'])
    plt.ylabel('Total KB')
    plt.title('Retained evidence by file type')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No retained artifacts/results found.')


In [ ]:
# Threshold / coverage trade-off when a result table contains confidence or probability
for path, table in result_tables:
    conf_cols = [c for c in table.columns if any(token in str(c).lower() for token in ('confidence','probability','proba','score','risk'))]
    correct_cols = [c for c in table.columns if 'correct' in str(c).lower()]
    if not conf_cols or not len(table):
        continue
    confidence = pd.to_numeric(table[conf_cols[0]], errors='coerce')
    valid_conf = confidence.notna()
    if valid_conf.sum() < 20:
        continue
    trade_rows = []
    for threshold in np.linspace(float(confidence[valid_conf].quantile(0.10)), float(confidence[valid_conf].quantile(0.90)), 9):
        accepted = valid_conf & (confidence >= threshold)
        row = {'threshold':float(threshold), 'coverage':float(accepted.mean()), 'review_rate':float((valid_conf & ~accepted).sum()/valid_conf.sum()), 'accepted_rows':int(accepted.sum())}
        if correct_cols:
            correctness = table[correct_cols[0]].astype(bool)
            row['accepted_accuracy'] = float(correctness[accepted].mean()) if accepted.any() else np.nan
        trade_rows.append(row)
    trade = pd.DataFrame(trade_rows)
    print('Trade-off table from', path.name, 'using', conf_cols[0])
    display(trade.round(4))
    plt.figure(figsize=(8,4))
    plt.plot(trade['threshold'], trade['coverage'], marker='o', label='coverage')
    if 'accepted_accuracy' in trade:
        plt.plot(trade['threshold'], trade['accepted_accuracy'], marker='o', label='accepted accuracy')
    plt.xlabel('Threshold')
    plt.ylabel('Rate')
    plt.title(f'Threshold trade-off — {path.name}')
    plt.legend()
    plt.tight_layout()
    plt.show()
    break


In [ ]:
# Produce a concise evidence-backed decision memo inside the notebook
project_summary = {
    'project': PROJECT_SLUG,
    'local_data_or_evidence_files': int(len(candidate_files)),
    'result_tables': int(len(result_tables)),
    'json_evidence_files': int(len(json_files)),
    'visual_evidence_files': int(len(png_files)),
    'has_tests': bool((PROJECT/'tests').exists() and any((PROJECT/'tests').rglob('test*.py'))),
    'has_readme': bool((PROJECT/'README.md').exists()),
}
if df is not None:
    project_summary.update({'inspected_rows':int(len(df)), 'inspected_columns':int(df.shape[1]), 'duplicate_rows':int(df.duplicated().sum()), 'missing_cells':int(df.isna().sum().sum())})
summary_table = pd.DataFrame({'item':list(project_summary.keys()), 'value':list(project_summary.values())})
display(summary_table)
print('DECISION PRINCIPLE')
print('1. Use the measured evidence above, not model complexity, to choose the final approach.')
print('2. Inspect the worst slices/failures before making a business or operational recommendation.')
print('3. Keep uncertain, novel or high-impact cases on a review/escalation path where appropriate.')
print('4. Treat the documented limitations as part of the solution, not as boilerplate.')


# Engineering appendix — canonical application source

The analysis and visual evidence come first. The cells below preserve additional canonical Python from this project for reviewers who want to inspect pipelines, APIs, tests, feature code, monitoring and reusable implementation details.


## Canonical source: `run.py`


In [ ]:
"""Linear Regression — Building Energy Efficiency Decision Model.

This file intentionally reads like a notebook/script rather than a framework full of
helper functions. The goal is to make the actual junior/graduate data-science work
visible: data loading, validation, EDA, modelling, diagnostics, uncertainty and a
usable decision layer.
"""
from __future__ import annotations

import argparse
import json
import math
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LassoCV, LinearRegression, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, learning_curve, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
from ucimlrepo import fetch_ucirepo

warnings.filterwarnings("ignore")

parser = argparse.ArgumentParser(description="Building heating-load regression portfolio project")
parser.add_argument("--output-dir", default="results", help="Directory for metrics, tables and plots")
parser.add_argument("--seed", type=int, default=42)
parser.add_argument("--test-size", type=float, default=0.20)
parser.add_argument("--bootstrap", type=int, default=250, help="Bootstrap refits for uncertainty summary")
parser.add_argument("--no-plots", action="store_true")
args = parser.parse_args()

RNG = np.random.default_rng(args.seed)
PROJECT = Path(__file__).resolve().parent
OUTPUT = PROJECT / args.output_dir
OUTPUT.mkdir(parents=True, exist_ok=True)

print("=" * 88)
print("LINEAR REGRESSION — BUILDING ENERGY EFFICIENCY DECISION MODEL")
print("=" * 88)
print("Project:", PROJECT)
print("Output:", OUTPUT)
print("Seed:", args.seed)

# -----------------------------------------------------------------------------
# 1. DATA ACQUISITION AND PROVENANCE
# -----------------------------------------------------------------------------
print("\n[1/12] Loading UCI Energy Efficiency dataset (ID 242)...")
energy = fetch_ucirepo(id=242)
X_raw = energy.data.features.copy()
y_raw = energy.data.targets.copy()

print("Dataset name:", energy.metadata.get("name", "Energy Efficiency"))
print("UCI id:", energy.metadata.get("uci_id", 242))
print("Rows:", len(X_raw))
print("Feature columns:", list(X_raw.columns))
print("Target columns:", list(y_raw.columns))

feature_names = [
    "relative_compactness",
    "surface_area",
    "wall_area",
    "roof_area",
    "overall_height",
    "orientation",
    "glazing_area",
    "glazing_area_distribution",
]

target_names = ["heating_load", "cooling_load"]

if X_raw.shape[1] != 8:
    raise ValueError(f"Expected 8 features from UCI Energy Efficiency, received {X_raw.shape[1]}")
if y_raw.shape[1] < 2:
    raise ValueError(f"Expected two targets from UCI Energy Efficiency, received {y_raw.shape[1]}")

X_raw.columns = feature_names
y_raw = y_raw.iloc[:, :2].copy()
y_raw.columns = target_names

df = pd.concat([X_raw, y_raw], axis=1)

print("\nPreview:")
print(df.head().to_string(index=False))

# -----------------------------------------------------------------------------
# 2. DATA QUALITY AUDIT
# -----------------------------------------------------------------------------
print("\n[2/12] Running schema and quality audit...")

expected_columns = feature_names + target_names
missing_columns = sorted(set(expected_columns) - set(df.columns))
unexpected_columns = sorted(set(df.columns) - set(expected_columns))
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")
if unexpected_columns:
    print("Unexpected columns:", unexpected_columns)

numeric_check = df[expected_columns].apply(pd.to_numeric, errors="coerce")
coercion_failures = numeric_check.isna().sum() - df[expected_columns].isna().sum()
if (coercion_failures > 0).any():
    raise ValueError(f"Non-numeric values detected: {coercion_failures[coercion_failures > 0].to_dict()}")

df = numeric_check.copy()

quality = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (100.0 * df.isna().mean()).round(3),
        "unique": df.nunique(dropna=False),
        "min": df.min(numeric_only=True),
        "max": df.max(numeric_only=True),
    }
)
quality.to_csv(OUTPUT / "data_audit.csv")

print(quality.to_string())
print("Duplicate rows:", int(df.duplicated().sum()))

if df.isna().any().any():
    print("Missing values are present and will be handled by the preprocessing pipeline.")
else:
    print("Published UCI table contains no missing values in this load.")

if len(df) != 768:
    print(f"WARNING: UCI documentation describes 768 rows; current load contains {len(df)} rows.")

# Plausibility checks derived from the published schema.
if not df["relative_compactness"].between(0, 1.5).all():
    raise ValueError("Relative compactness outside a broad plausible range")
if not df["glazing_area"].between(0, 1).all():
    raise ValueError("Glazing area outside [0, 1]")
if (df["heating_load"] <= 0).any() or (df["cooling_load"] <= 0).any():
    raise ValueError("Energy loads should be positive")

# -----------------------------------------------------------------------------
# 3. EXPLORATORY DATA ANALYSIS
# -----------------------------------------------------------------------------
print("\n[3/12] Exploratory analysis...")

summary = df.describe(include="all").T
summary.to_csv(OUTPUT / "descriptive_statistics.csv")
print(summary.round(3).to_string())

corr = df.corr(numeric_only=True)
corr.to_csv(OUTPUT / "correlation_matrix.csv")

heating_quantiles = df["heating_load"].quantile([0.05, 0.25, 0.50, 0.75, 0.95])
print("\nHeating-load quantiles:")
print(heating_quantiles.round(3).to_string())

orientation_summary = (
    df.groupby("orientation", as_index=False)
    .agg(
        rows=("heating_load", "size"),
        heating_mean=("heating_load", "mean"),
        heating_median=("heating_load", "median"),
        cooling_mean=("cooling_load", "mean"),
    )
    .sort_values("orientation")
)
orientation_summary.to_csv(OUTPUT / "orientation_summary.csv", index=False)
print("\nLoad by orientation:")
print(orientation_summary.round(3).to_string(index=False))

glazing_summary = (
    df.groupby("glazing_area", as_index=False)
    .agg(
        rows=("heating_load", "size"),
        heating_mean=("heating_load", "mean"),
        heating_std=("heating_load", "std"),
        cooling_mean=("cooling_load", "mean"),
    )
    .sort_values("glazing_area")
)
glazing_summary.to_csv(OUTPUT / "glazing_summary.csv", index=False)

height_summary = (
    df.groupby("overall_height", as_index=False)
    .agg(
        rows=("heating_load", "size"),
        heating_mean=("heating_load", "mean"),
        cooling_mean=("cooling_load", "mean"),
    )
    .sort_values("overall_height")
)
height_summary.to_csv(OUTPUT / "height_summary.csv", index=False)

if not args.no_plots:
    plt.figure(figsize=(8, 5))
    plt.hist(df["heating_load"], bins=28, alpha=0.80)
    plt.axvline(df["heating_load"].median(), linestyle="--", linewidth=2, label=f"median={df['heating_load'].median():.2f}")
    plt.axvline(df["heating_load"].quantile(0.75), linestyle=":", linewidth=2, label=f"75th pct={df['heating_load'].quantile(0.75):.2f}")
    plt.title("Heating-load distribution")
    plt.xlabel("Heating Load")
    plt.ylabel("Buildings")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT / "01_heating_load_distribution.png", dpi=160)
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.scatter(df["heating_load"], df["cooling_load"], alpha=0.45, s=25)
    plt.xlabel("Heating Load")
    plt.ylabel("Cooling Load")
    plt.title("Heating vs cooling load")
    plt.tight_layout()
    plt.savefig(OUTPUT / "02_heating_vs_cooling.png", dpi=160)
    plt.close()

    plt.figure(figsize=(10, 8))
    image = plt.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
    plt.colorbar(image, label="Correlation")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=70, ha="right")
    plt.yticks(range(len(corr.index)), corr.index)
    plt.title("Correlation matrix")
    plt.tight_layout()
    plt.savefig(OUTPUT / "03_correlation_matrix.png", dpi=160)
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.scatter(df["relative_compactness"], df["heating_load"], alpha=0.45, s=25)
    plt.xlabel("Relative compactness")
    plt.ylabel("Heating Load")
    plt.title("Heating load vs relative compactness")
    plt.tight_layout()
    plt.savefig(OUTPUT / "04_compactness_vs_heating.png", dpi=160)
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.scatter(df["surface_area"], df["heating_load"], alpha=0.45, s=25)
    plt.xlabel("Surface area")
    plt.ylabel("Heating Load")
    plt.title("Heating load vs surface area")
    plt.tight_layout()
    plt.savefig(OUTPUT / "05_surface_area_vs_heating.png", dpi=160)
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.scatter(df["glazing_area"], df["heating_load"], alpha=0.45, s=25)
    plt.xlabel("Glazing area")
    plt.ylabel("Heating Load")
    plt.title("Heating load vs glazing area")
    plt.tight_layout()
    plt.savefig(OUTPUT / "06_glazing_vs_heating.png", dpi=160)
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.bar(orientation_summary["orientation"].astype(str), orientation_summary["heating_mean"])
    plt.xlabel("Orientation")
    plt.ylabel("Mean Heating Load")
    plt.title("Average heating load by orientation")
    plt.tight_layout()
    plt.savefig(OUTPUT / "07_orientation_heating.png", dpi=160)
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.bar(glazing_summary["glazing_area"].astype(str), glazing_summary["heating_mean"])
    plt.xlabel("Glazing area")
    plt.ylabel("Mean Heating Load")
    plt.title("Average heating load by glazing area")
    plt.tight_layout()
    plt.savefig(OUTPUT / "08_glazing_group_heating.png", dpi=160)
    plt.close()

# -----------------------------------------------------------------------------
# 4. TRAIN / TEST DESIGN
# -----------------------------------------------------------------------------
print("\n[4/12] Creating holdout split and preprocessing...")

feature_columns = feature_names
categorical_columns = ["orientation", "glazing_area_distribution"]
numeric_columns = [c for c in feature_columns if c not in categorical_columns]

target = "heating_load"
X = df[feature_columns].copy()
y = df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=args.test_size,
    random_state=args.seed,
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train target mean:", round(float(y_train.mean()), 3))
print("Test target mean:", round(float(y_test.mean()), 3))

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", drop=None, sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_columns),
        ("categorical", categorical_pipeline, categorical_columns),
    ],
    remainder="drop",
)

# -----------------------------------------------------------------------------
# 5. BASELINE + ORDINARY LINEAR REGRESSION
# -----------------------------------------------------------------------------
print("\n[5/12] Training baseline and ordinary LinearRegression...")

baseline = DummyRegressor(strategy="median")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

linear_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", LinearRegression()),
    ]
)
linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)

# -----------------------------------------------------------------------------
# 6. REGULARISED AND NON-LINEAR ALTERNATIVES
# -----------------------------------------------------------------------------
print("\n[6/12] Training Ridge, Lasso and polynomial alternatives...")

ridge_alphas = np.logspace(-4, 4, 80)
ridge_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", RidgeCV(alphas=ridge_alphas)),
    ]
)
ridge_model.fit(X_train, y_train)
ridge_pred = ridge_model.predict(X_test)

lasso_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", LassoCV(alphas=np.logspace(-4, 1, 80), cv=5, random_state=args.seed, max_iter=100_000)),
    ]
)
lasso_model.fit(X_train, y_train)
lasso_pred = lasso_model.predict(X_test)

poly_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
                    ("scale", StandardScaler()),
                ]
            ),
            numeric_columns,
        ),
        ("categorical", categorical_pipeline, categorical_columns),
    ],
    remainder="drop",
)

poly_model = Pipeline(
    steps=[
        ("preprocess", poly_preprocessor),
        ("model", RidgeCV(alphas=ridge_alphas)),
    ]
)
poly_model.fit(X_train, y_train)
poly_pred = poly_model.predict(X_test)

# -----------------------------------------------------------------------------
# 7. HOLDOUT METRICS
# -----------------------------------------------------------------------------
print("\n[7/12] Comparing holdout performance...")

prediction_map = {
    "dummy_median": baseline_pred,
    "linear_regression": linear_pred,
    "ridge": ridge_pred,
    "lasso": lasso_pred,
    "polynomial_ridge": poly_pred,
}

metric_rows = []
for model_name, pred in prediction_map.items():
    mae = mean_absolute_error(y_test, pred)
    rmse = math.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    mape = mean_absolute_percentage_error(y_test, pred)
    metric_rows.append(
        {
            "model": model_name,
            "mae": float(mae),
            "rmse": float(rmse),
            "r2": float(r2),
            "mape": float(mape),
        }
    )

metrics_df = pd.DataFrame(metric_rows).sort_values("rmse").reset_index(drop=True)
print(metrics_df.round(4).to_string(index=False))
metrics_df.to_csv(OUTPUT / "model_comparison.csv", index=False)

best_model_name = str(metrics_df.iloc[0]["model"])
model_lookup = {
    "dummy_median": baseline,
    "linear_regression": linear_model,
    "ridge": ridge_model,
    "lasso": lasso_model,
    "polynomial_ridge": poly_model,
}
best_model = model_lookup[best_model_name]
best_pred = prediction_map[best_model_name]

linear_metrics = metrics_df.loc[metrics_df["model"] == "linear_regression"].iloc[0]
baseline_metrics = metrics_df.loc[metrics_df["model"] == "dummy_median"].iloc[0]

improvement_vs_baseline = 100.0 * (baseline_metrics["rmse"] - linear_metrics["rmse"]) / baseline_metrics["rmse"]
print(f"\nLinear Regression RMSE improvement vs median baseline: {improvement_vs_baseline:.2f}%")
print("Best holdout model:", best_model_name)

if not args.no_plots:
    plt.figure(figsize=(8, 5))
    plt.bar(metrics_df["model"], metrics_df["rmse"])
    plt.ylabel("RMSE")
    plt.title("Model comparison — lower is better")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(OUTPUT / "09_model_rmse_comparison.png", dpi=160)
    plt.close()

    plt.figure(figsize=(7, 6))
    plt.scatter(y_test, linear_pred, alpha=0.65, s=28)
    lower = min(float(y_test.min()), float(linear_pred.min()))
    upper = max(float(y_test.max()), float(linear_pred.max()))
    plt.plot([lower, upper], [lower, upper], linestyle="--")
    plt.xlabel("Actual Heating Load")
    plt.ylabel("Predicted Heating Load")
    plt.title("Ordinary Linear Regression — actual vs predicted")
    plt.tight_layout()
    plt.savefig(OUTPUT / "10_linear_actual_vs_predicted.png", dpi=160)
    plt.close()

# -----------------------------------------------------------------------------
# 8. CROSS-VALIDATION AND LEARNING CURVE
# -----------------------------------------------------------------------------
print("\n[8/12] Cross-validation and learning-curve diagnostics...")

cv = KFold(n_splits=5, shuffle=True, random_state=args.seed)
cv_rows = []
for model_name, model in {
    "linear_regression": linear_model,
    "ridge": ridge_model,
    "polynomial_ridge": poly_model,
}.items():
    neg_rmse = cross_val_score(model, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=None)
    cv_rows.append(
        {
            "model": model_name,
            "cv_rmse_mean": float(-neg_rmse.mean()),
            "cv_rmse_std": float(neg_rmse.std()),
            "fold_rmse": [float(-x) for x in neg_rmse],
        }
    )

cv_df = pd.DataFrame(
    [{k: v for k, v in row.items() if k != "fold_rmse"} for row in cv_rows]
).sort_values("cv_rmse_mean")
cv_df.to_csv(OUTPUT / "cross_validation.csv", index=False)
print(cv_df.round(4).to_string(index=False))

train_sizes, train_scores, valid_scores = learning_curve(
    linear_model,
    X_train,
    y_train,
    train_sizes=np.linspace(0.20, 1.0, 6),
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=None,
)

learning_df = pd.DataFrame(
    {
        "train_rows": train_sizes,
        "train_rmse": -train_scores.mean(axis=1),
        "validation_rmse": -valid_scores.mean(axis=1),
        "validation_rmse_std": valid_scores.std(axis=1),
    }
)
learning_df.to_csv(OUTPUT / "learning_curve.csv", index=False)
print("\nLinear Regression learning curve:")
print(learning_df.round(4).to_string(index=False))

if not args.no_plots:
    plt.figure(figsize=(8, 5))
    plt.plot(learning_df["train_rows"], learning_df["train_rmse"], marker="o", label="train RMSE")
    plt.plot(learning_df["train_rows"], learning_df["validation_rmse"], marker="o", label="validation RMSE")
    plt.xlabel("Training rows")
    plt.ylabel("RMSE")
    plt.title("Linear Regression learning curve")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT / "11_linear_learning_curve.png", dpi=160)
    plt.close()

# -----------------------------------------------------------------------------
# 9. RESIDUAL AND ERROR ANALYSIS
# -----------------------------------------------------------------------------
print("\n[9/12] Residual diagnostics and error slices...")

predictions = X_test.reset_index(drop=False).rename(columns={"index": "source_index"})
predictions["actual_heating_load"] = y_test.reset_index(drop=True)
predictions["linear_prediction"] = linear_pred
predictions["best_prediction"] = best_pred
predictions["linear_residual"] = predictions["actual_heating_load"] - predictions["linear_prediction"]
predictions["best_residual"] = predictions["actual_heating_load"] - predictions["best_prediction"]
predictions["linear_abs_error"] = predictions["linear_residual"].abs()
predictions["best_abs_error"] = predictions["best_residual"].abs()
predictions.to_csv(OUTPUT / "predictions.csv", index=False)

print("Linear residual mean:", round(float(predictions["linear_residual"].mean()), 4))
print("Linear residual std:", round(float(predictions["linear_residual"].std()), 4))
print("Linear worst absolute error:", round(float(predictions["linear_abs_error"].max()), 4))

worst = predictions.nlargest(12, "linear_abs_error")
worst.to_csv(OUTPUT / "worst_linear_errors.csv", index=False)
print("\nWorst Linear Regression cases:")
print(
    worst[
        [
            "source_index",
            "actual_heating_load",
            "linear_prediction",
            "linear_abs_error",
            "relative_compactness",
            "surface_area",
            "overall_height",
            "glazing_area",
            "orientation",
        ]
    ].round(3).to_string(index=False)
)

orientation_error = (
    predictions.groupby("orientation", as_index=False)
    .agg(
        rows=("linear_abs_error", "size"),
        linear_mae=("linear_abs_error", "mean"),
        best_mae=("best_abs_error", "mean"),
        residual_bias=("linear_residual", "mean"),
    )
    .sort_values("linear_mae", ascending=False)
)
orientation_error.to_csv(OUTPUT / "error_by_orientation.csv", index=False)

predictions["glazing_band"] = pd.cut(
    predictions["glazing_area"],
    bins=[-np.inf, 0.0, 0.15, 0.30, np.inf],
    labels=["none", "low", "medium", "high"],
)
glazing_error = (
    predictions.groupby("glazing_band", observed=False, as_index=False)
    .agg(
        rows=("linear_abs_error", "size"),
        linear_mae=("linear_abs_error", "mean"),
        best_mae=("best_abs_error", "mean"),
        residual_bias=("linear_residual", "mean"),
    )
)
glazing_error.to_csv(OUTPUT / "error_by_glazing_band.csv", index=False)

predictions["compactness_band"] = pd.qcut(
    predictions["relative_compactness"],
    q=4,
    duplicates="drop",
)
compactness_error = (
    predictions.groupby("compactness_band", observed=False, as_index=False)
    .agg(
        rows=("linear_abs_error", "size"),
        linear_mae=("linear_abs_error", "mean"),
        best_mae=("best_abs_error", "mean"),
        residual_bias=("linear_residual", "mean"),
    )
)
compactness_error.to_csv(OUTPUT / "error_by_compactness.csv", index=False)

print("\nLinear error by orientation:")
print(orientation_error.round(4).to_string(index=False))
print("\nLinear error by glazing band:")
print(glazing_error.round(4).to_string(index=False))

if not args.no_plots:
    plt.figure(figsize=(8, 5))
    plt.scatter(predictions["linear_prediction"], predictions["linear_residual"], alpha=0.60, s=26)
    plt.axhline(0, linestyle="--", linewidth=1.5)
    plt.xlabel("Linear Regression prediction")
    plt.ylabel("Residual (actual - predicted)")
    plt.title("Residuals vs predicted values")
    plt.tight_layout()
    plt.savefig(OUTPUT / "12_linear_residuals_vs_prediction.png", dpi=160)
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.hist(predictions["linear_residual"], bins=25, alpha=0.82)
    plt.axvline(0, linestyle="--", linewidth=1.5)
    plt.xlabel("Residual")
    plt.ylabel("Rows")
    plt.title("Linear Regression residual distribution")
    plt.tight_layout()
    plt.savefig(OUTPUT / "13_linear_residual_distribution.png", dpi=160)
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.bar(orientation_error["orientation"].astype(str), orientation_error["linear_mae"])
    plt.xlabel("Orientation")
    plt.ylabel("Linear Regression MAE")
    plt.title("Error slice by orientation")
    plt.tight_layout()
    plt.savefig(OUTPUT / "14_error_by_orientation.png", dpi=160)
    plt.close()

# -----------------------------------------------------------------------------
# 10. COEFFICIENT INTERPRETATION
# -----------------------------------------------------------------------------
print("\n[10/12] Inspecting Linear Regression coefficients...")

preprocess_fitted = linear_model.named_steps["preprocess"]
model_fitted = linear_model.named_steps["model"]

numeric_names = numeric_columns
categorical_encoder = preprocess_fitted.named_transformers_["categorical"].named_steps["onehot"]
categorical_names = categorical_encoder.get_feature_names_out(categorical_columns).tolist()
transformed_feature_names = numeric_names + categorical_names

if len(transformed_feature_names) != len(model_fitted.coef_):
    raise RuntimeError("Transformed feature names do not align with LinearRegression coefficients")

coefficients = pd.DataFrame(
    {
        "feature": transformed_feature_names,
        "coefficient": model_fitted.coef_.astype(float),
    }
)
coefficients["abs_coefficient"] = coefficients["coefficient"].abs()
coefficients = coefficients.sort_values("abs_coefficient", ascending=False).reset_index(drop=True)
coefficients.to_csv(OUTPUT / "coefficients.csv", index=False)

print("Linear intercept:", round(float(model_fitted.intercept_), 4))
print("\nLargest absolute coefficients after preprocessing:")
print(coefficients.head(20).round(4).to_string(index=False))

if not args.no_plots:
    coef_plot = coefficients.head(15).sort_values("coefficient")
    plt.figure(figsize=(9, 6))
    plt.barh(coef_plot["feature"], coef_plot["coefficient"])
    plt.xlabel("Coefficient")
    plt.title("Linear Regression coefficient magnitude")
    plt.tight_layout()
    plt.savefig(OUTPUT / "15_linear_coefficients.png", dpi=160)
    plt.close()

print(
    "\nInterpretation warning: coefficients describe conditional associations in this simulated dataset. "
    "Correlated geometric variables and preprocessing mean they should not be presented as causal effects."
)

# -----------------------------------------------------------------------------
# 11. BOOTSTRAP UNCERTAINTY
# -----------------------------------------------------------------------------
print("\n[11/12] Bootstrap uncertainty for Linear Regression...")

bootstrap_rmses = []
bootstrap_maes = []
bootstrap_prediction_matrix = []

bootstrap_iterations = max(25, int(args.bootstrap))
for iteration in range(bootstrap_iterations):
    sample_positions = RNG.integers(0, len(X_train), size=len(X_train))
    X_boot = X_train.iloc[sample_positions]
    y_boot = y_train.iloc[sample_positions]

    boot_model = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", LinearRegression()),
        ]
    )
    boot_model.fit(X_boot, y_boot)
    boot_pred = boot_model.predict(X_test)

    bootstrap_rmses.append(math.sqrt(mean_squared_error(y_test, boot_pred)))
    bootstrap_maes.append(mean_absolute_error(y_test, boot_pred))
    bootstrap_prediction_matrix.append(boot_pred)

bootstrap_prediction_matrix = np.vstack(bootstrap_prediction_matrix)
predictions["linear_bootstrap_p05"] = np.quantile(bootstrap_prediction_matrix, 0.05, axis=0)
predictions["linear_bootstrap_p50"] = np.quantile(bootstrap_prediction_matrix, 0.50, axis=0)
predictions["linear_bootstrap_p95"] = np.quantile(bootstrap_prediction_matrix, 0.95, axis=0)
predictions["linear_bootstrap_width"] = predictions["linear_bootstrap_p95"] - predictions["linear_bootstrap_p05"]
predictions.to_csv(OUTPUT / "predictions.csv", index=False)

bootstrap_summary = {
    "iterations": bootstrap_iterations,
    "rmse_mean": float(np.mean(bootstrap_rmses)),
    "rmse_p05": float(np.quantile(bootstrap_rmses, 0.05)),
    "rmse_p95": float(np.quantile(bootstrap_rmses, 0.95)),
    "mae_mean": float(np.mean(bootstrap_maes)),
    "mae_p05": float(np.quantile(bootstrap_maes, 0.05)),
    "mae_p95": float(np.quantile(bootstrap_maes, 0.95)),
    "mean_prediction_interval_width": float(predictions["linear_bootstrap_width"].mean()),
}
(OUTPUT / "bootstrap_summary.json").write_text(json.dumps(bootstrap_summary, indent=2), encoding="utf-8")
print(json.dumps(bootstrap_summary, indent=2))

if not args.no_plots:
    plt.figure(figsize=(8, 5))
    plt.hist(bootstrap_rmses, bins=25, alpha=0.82)
    plt.axvline(np.mean(bootstrap_rmses), linestyle="--", linewidth=2, label=f"mean={np.mean(bootstrap_rmses):.3f}")
    plt.xlabel("Bootstrap RMSE")
    plt.ylabel("Refits")
    plt.title("Linear Regression bootstrap RMSE")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT / "16_bootstrap_rmse.png", dpi=160)
    plt.close()

# -----------------------------------------------------------------------------
# 12. SCENARIO ANALYSIS + DECISION POLICY
# -----------------------------------------------------------------------------
print("\n[12/12] Building scenario analysis and decision policy...")

load_review_threshold = float(y_train.quantile(0.75))

scenario_rows = []
compactness_values = sorted(set(np.quantile(df["relative_compactness"], [0.15, 0.50, 0.85]).round(3)))
surface_values = sorted(set(np.quantile(df["surface_area"], [0.20, 0.50, 0.80]).round(3)))
glazing_values = sorted(df["glazing_area"].unique())
orientation_values = sorted(df["orientation"].unique())

base = X_train.median(numeric_only=True).to_dict()
base["orientation"] = float(X_train["orientation"].mode().iloc[0])
base["glazing_area_distribution"] = float(X_train["glazing_area_distribution"].mode().iloc[0])

for compactness in compactness_values:
    for surface in surface_values:
        for glazing in glazing_values:
            for orientation in orientation_values:
                row = dict(base)
                row["relative_compactness"] = float(compactness)
                row["surface_area"] = float(surface)
                row["glazing_area"] = float(glazing)
                row["orientation"] = float(orientation)
                scenario_rows.append(row)

scenarios = pd.DataFrame(scenario_rows)[feature_columns]
scenarios["linear_prediction"] = linear_model.predict(scenarios)
scenarios["ridge_prediction"] = ridge_model.predict(scenarios)
scenarios["best_prediction"] = best_model.predict(scenarios)
scenarios["model_spread"] = scenarios[["linear_prediction", "ridge_prediction", "best_prediction"]].max(axis=1) - scenarios[["linear_prediction", "ridge_prediction", "best_prediction"]].min(axis=1)
scenarios["decision"] = np.where(
    scenarios["linear_prediction"] >= load_review_threshold,
    "HIGH LOAD - REVIEW",
    "LOW / NORMAL LOAD",
)
scenarios = scenarios.sort_values(["linear_prediction", "model_spread"], ascending=[False, False]).reset_index(drop=True)
scenarios.to_csv(OUTPUT / "scenario_analysis.csv", index=False)

print("Heating-load review threshold (training 75th percentile):", round(load_review_threshold, 3))
print("\nHighest predicted heating-load scenarios:")
print(scenarios.head(15).round(3).to_string(index=False))

high_load_share = float((scenarios["decision"] == "HIGH LOAD - REVIEW").mean())
print(f"Scenario grid flagged for review: {100 * high_load_share:.1f}%")

# A concrete single-design inference example.
example_design = X_train.median(numeric_only=True).to_dict()
example_design["orientation"] = float(X_train["orientation"].mode().iloc[0])
example_design["glazing_area_distribution"] = float(X_train["glazing_area_distribution"].mode().iloc[0])
example = pd.DataFrame([example_design])[feature_columns]
example_linear = float(linear_model.predict(example)[0])
example_best = float(best_model.predict(example)[0])
example_decision = "HIGH LOAD - REVIEW" if example_linear >= load_review_threshold else "LOW / NORMAL LOAD"

print("\nExample design:")
print(example.round(3).to_string(index=False))
print("Linear Regression heating-load estimate:", round(example_linear, 3))
print("Best-model heating-load estimate:", round(example_best, 3))
print("Decision:", example_decision)

if not args.no_plots:
    scenario_plot = scenarios.groupby("glazing_area", as_index=False)["linear_prediction"].mean()
    plt.figure(figsize=(8, 5))
    plt.plot(scenario_plot["glazing_area"], scenario_plot["linear_prediction"], marker="o")
    plt.axhline(load_review_threshold, linestyle="--", label="review threshold")
    plt.xlabel("Glazing area")
    plt.ylabel("Mean predicted Heating Load")
    plt.title("Scenario sensitivity — glazing area")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT / "17_scenario_glazing_sensitivity.png", dpi=160)
    plt.close()

# -----------------------------------------------------------------------------
# RETAIN MODEL AND MACHINE-READABLE EVIDENCE
# -----------------------------------------------------------------------------
joblib.dump(linear_model, OUTPUT / "linear_regression.joblib")
joblib.dump(best_model, OUTPUT / "best_model.joblib")

metrics_payload = {
    "dataset": {
        "name": "UCI Energy Efficiency",
        "uci_id": 242,
        "rows": int(len(df)),
        "features": int(len(feature_columns)),
        "primary_target": target,
        "licence": "CC BY 4.0",
        "doi": "10.24432/C51307",
    },
    "split": {
        "train_rows": int(len(X_train)),
        "test_rows": int(len(X_test)),
        "test_size": float(args.test_size),
        "random_seed": int(args.seed),
    },
    "models": {
        row["model"]: {
            "mae": row["mae"],
            "rmse": row["rmse"],
            "r2": row["r2"],
            "mape": row["mape"],
        }
        for row in metric_rows
    },
    "linear_regression": {
        "rmse_improvement_vs_dummy_pct": float(improvement_vs_baseline),
        "intercept": float(model_fitted.intercept_),
        "residual_mean": float(predictions["linear_residual"].mean()),
        "residual_std": float(predictions["linear_residual"].std()),
    },
    "regularisation": {
        "ridge_alpha": float(ridge_model.named_steps["model"].alpha_),
        "lasso_alpha": float(lasso_model.named_steps["model"].alpha_),
        "polynomial_ridge_alpha": float(poly_model.named_steps["model"].alpha_),
    },
    "cross_validation": {
        row["model"]: {
            "rmse_mean": row["cv_rmse_mean"],
            "rmse_std": row["cv_rmse_std"],
            "fold_rmse": row["fold_rmse"],
        }
        for row in cv_rows
    },
    "bootstrap": bootstrap_summary,
    "decision": {
        "heating_load_review_threshold": load_review_threshold,
        "scenario_high_load_share": high_load_share,
        "example_linear_prediction": example_linear,
        "example_best_prediction": example_best,
        "example_decision": example_decision,
    },
    "best_holdout_model": best_model_name,
    "limitations": [
        "The UCI data is simulated building-energy data, not a representative sample of all real buildings.",
        "Regression coefficients are associations conditional on correlated design variables, not causal effects.",
        "A production engineering decision would require external measured data, climate/location variables and domain validation.",
    ],
}

(OUTPUT / "metrics.json").write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")

print("\n" + "=" * 88)
print("PROJECT COMPLETE")
print("=" * 88)
print("Linear Regression holdout RMSE:", round(float(linear_metrics["rmse"]), 4))
print("Linear Regression holdout R²:", round(float(linear_metrics["r2"]), 4))
print("Best holdout model:", best_model_name)
print("Decision threshold:", round(load_review_threshold, 4))
print("Evidence written to:", OUTPUT)
print(
    "Final interpretation: ordinary Linear Regression is retained as the transparent baseline. "
    "Use the comparison models to test whether extra complexity materially improves the measured decision problem."
)


## Canonical source: `tests/test_linear_regression_project.py`


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


PROJECT = Path(__file__).resolve().parents[1]


def test_run_script_compiles() -> None:
    source = (PROJECT / "run.py").read_text(encoding="utf-8")
    compile(source, str(PROJECT / "run.py"), "exec")


def test_required_portfolio_evidence_is_in_source() -> None:
    source = (PROJECT / "run.py").read_text(encoding="utf-8")
    required = [
        "fetch_ucirepo(id=242)",
        "DummyRegressor",
        "LinearRegression",
        "RidgeCV",
        "LassoCV",
        "cross_val_score",
        "learning_curve",
        "linear_residual",
        "bootstrap",
        "scenario_analysis.csv",
        "metrics.json",
    ]
    for token in required:
        assert token in source, token


def test_linear_regression_recovers_a_simple_signal() -> None:
    rng = np.random.default_rng(42)
    x1 = rng.normal(size=300)
    x2 = rng.normal(size=300)
    y = 4.0 + 2.5 * x1 - 1.2 * x2 + rng.normal(scale=0.05, size=300)
    X = pd.DataFrame({"x1": x1, "x2": x2})
    model = LinearRegression().fit(X, y)
    pred = model.predict(X)
    assert r2_score(y, pred) > 0.99
    assert abs(model.coef_[0] - 2.5) < 0.05
    assert abs(model.coef_[1] + 1.2) < 0.05


def test_high_load_screening_rule() -> None:
    training_loads = pd.Series([10.0, 12.0, 15.0, 20.0, 25.0, 30.0, 35.0, 40.0])
    threshold = float(training_loads.quantile(0.75))
    predictions = np.array([15.0, threshold - 0.01, threshold, threshold + 5.0])
    decisions = np.where(predictions >= threshold, "HIGH LOAD - REVIEW", "LOW / NORMAL LOAD")
    assert decisions.tolist() == [
        "LOW / NORMAL LOAD",
        "LOW / NORMAL LOAD",
        "HIGH LOAD - REVIEW",
        "HIGH LOAD - REVIEW",
    ]


# Portfolio depth check

**Meaningful visible code lines after all notebook passes:** 1,584. The working target for a major application is roughly 1,000 meaningful lines when justified by the problem. This notebook is in/above the working depth range. Line count is never permission to add filler; depth must come from data, analysis, visualisation, modelling/engineering, evaluation, robustness and decision logic.
